In [42]:
import pandas as pd

fund_master = pd.read_csv("../data/processed/01_fund_master.csv")

sharpe_df = pd.read_csv("../reports/sharpe_ratio.csv")

In [3]:
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")

nav = pd.read_csv("../data/processed/02_nav_history.csv")

transactions = pd.read_csv("../data/processed/08_investor_transactions.csv")

portfolio = pd.read_csv("../data/processed/09_portfolio_holdings.csv")

benchmark = pd.read_csv("../data/processed/10_benchmark_indices.csv")

In [4]:
nav["date"] = pd.to_datetime(nav["date"])

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

portfolio["portfolio_date"] = pd.to_datetime(
    portfolio["portfolio_date"]
)

In [5]:
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [7]:
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

In [8]:
nav.head()

,amfi_code,date,nav,daily_return
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-0.010306
2,100016,2022-01-05,521.7239,0.012865
3,100016,2022-01-06,515.7880,-0.011377
4,100016,2022-01-07,515.1639,-0.001210


In [9]:
results = []

for fund in nav["amfi_code"].unique():

    df = nav[nav["amfi_code"] == fund].copy()

    returns = df["daily_return"].dropna()

    if len(returns) == 0:
        continue

    # Historical VaR (95%)
    var95 = np.percentile(returns, 5)

    # Conditional VaR (Expected Shortfall)
    cvar95 = returns[returns <= var95].mean()

    results.append({
        "amfi_code": fund,
        "VaR_95": round(var95, 6),
        "CVaR_95": round(cvar95, 6)
    })

var_cvar_df = pd.DataFrame(results)

var_cvar_df.head()

,amfi_code,VaR_95,CVaR_95
0,100016,-0.014364,-0.018060
1,100025,-0.003793,-0.004994
2,100033,-0.019034,-0.023456
3,101206,-0.013282,-0.017439
4,101207,-0.026021,-0.032459


In [10]:
var_cvar_df = var_cvar_df.sort_values(
    "VaR_95"
)

var_cvar_df.head(10)

,amfi_code,VaR_95,CVaR_95
22,119599,-0.026859,-0.032384
17,119095,-0.026188,-0.031667
4,101207,-0.026021,-0.032459
11,118634,-0.025438,-0.032304
21,119598,-0.024507,-0.030595
39,149324,-0.023483,-0.031036
7,102886,-0.019220,-0.023251
2,100033,-0.019034,-0.023456
25,120505,-0.018892,-0.024342
16,119094,-0.018480,-0.024260


In [11]:
var_cvar_df.to_csv(
    "../reports/var_cvar_report.csv",
    index=False
)

In [12]:
import plotly.express as px

top10 = var_cvar_df.nsmallest(10, "VaR_95")

fig = px.bar(
    top10,
    x="amfi_code",
    y="VaR_95",
    color="VaR_95",
    title="Top 10 Funds by Historical VaR (95%)",
    text="VaR_95"
)

fig.update_layout(
    xaxis_title="AMFI Code",
    yaxis_title="VaR (95%)",
    title_x=0.5
)

fig.show()

In [13]:
fig.write_image(
    "../reports/var95_chart.png",
    width=1600,
    height=800
)

In [14]:
nav.columns

Index(['amfi_code', 'date', 'nav', 'daily_return'], dtype='str')

In [15]:
top5_funds = nav["amfi_code"].unique()[:5]

top5_funds

array([100016, 100025, 100033, 101206, 101207])

In [16]:
import numpy as np

rolling_list = []

for fund in top5_funds:

    df = (
        nav[nav["amfi_code"] == fund]
        .sort_values("date")
        .copy()
    )

    df["Rolling Sharpe"] = (
        df["daily_return"].rolling(90).mean()
        /
        df["daily_return"].rolling(90).std()
    ) * np.sqrt(252)

    rolling_list.append(df)

rolling_df = pd.concat(rolling_list)

rolling_df.head()

,amfi_code,date,nav,daily_return,Rolling Sharpe
0,100016,2022-01-03,520.4608,NaN,NaN
1,100016,2022-01-04,515.0971,-0.010306,NaN
2,100016,2022-01-05,521.7239,0.012865,NaN
3,100016,2022-01-06,515.7880,-0.011377,NaN
4,100016,2022-01-07,515.1639,-0.001210,NaN


In [17]:
import plotly.express as px

fig = px.line(
    rolling_df,
    x="date",
    y="Rolling Sharpe",
    color="amfi_code",
    title="90-Day Rolling Sharpe Ratio"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Date",
    yaxis_title="Rolling Sharpe Ratio",
    legend_title="AMFI Code"
)

fig.show()

In [18]:
plot_df = rolling_df.copy()
plot_df["date"] = plot_df["date"].astype(str)

fig = px.line(
    plot_df,
    x="date",
    y="Rolling Sharpe",
    color="amfi_code",
    title="90-Day Rolling Sharpe Ratio"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Date",
    yaxis_title="Rolling Sharpe Ratio"
)

fig.write_image(
    "../reports/rolling_sharpe_chart.png",
    width=1600,
    height=800
)

In [19]:
rolling_df.to_csv(
    "../reports/rolling_sharpe.csv",
    index=False
)

In [20]:
transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [21]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

In [22]:
first_txn = (
    transactions
    .groupby("investor_id")["transaction_date"]
    .min()
    .reset_index()
)

first_txn.rename(
    columns={"transaction_date": "first_transaction"},
    inplace=True
)

first_txn["cohort_year"] = (
    first_txn["first_transaction"].dt.year
)

first_txn.head()

,investor_id,first_transaction,cohort_year
0,INV000001,2024-11-04,2024
1,INV000002,2024-03-29,2024
2,INV000003,2024-07-16,2024
3,INV000004,2024-03-16,2024
4,INV000005,2024-04-27,2024


In [23]:
transactions = transactions.merge(
    first_txn[["investor_id", "cohort_year"]],
    on="investor_id",
    how="left"
)

transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,cohort_year
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified,2024
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified,2024
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified,2024
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending,2024
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending,2024


In [24]:
sip_avg = (
    transactions[
        transactions["transaction_type"] == "SIP"
    ]
    .groupby("cohort_year")["amount_inr"]
    .mean()
    .reset_index()
)

sip_avg.rename(
    columns={"amount_inr": "Average SIP Amount"},
    inplace=True
)

sip_avg

,cohort_year,Average SIP Amount
0,2024,10996.885825
1,2025,13505.209581


In [25]:
total_investment = (
    transactions
    .groupby("cohort_year")["amount_inr"]
    .sum()
    .reset_index()
)

total_investment.rename(
    columns={"amount_inr": "Total Invested"},
    inplace=True
)

total_investment

,cohort_year,Total Invested
0,2024,3491125187
1,2025,30455243


In [26]:
top_fund = (
    transactions
    .groupby(["cohort_year", "amfi_code"])["amount_inr"]
    .sum()
    .reset_index()
)

top_fund = (
    top_fund
    .sort_values(
        ["cohort_year", "amount_inr"],
        ascending=[True, False]
    )
    .drop_duplicates("cohort_year")
)

top_fund.rename(
    columns={
        "amfi_code": "Top Fund",
        "amount_inr": "Fund Investment"
    },
    inplace=True
)

top_fund

,cohort_year,Top Fund,Fund Investment
6,2024,102885,100126141
62,2025,119599,1478507


In [27]:
cohort_df = (
    sip_avg
    .merge(
        total_investment,
        on="cohort_year"
    )
    .merge(
        top_fund[
            [
                "cohort_year",
                "Top Fund",
                "Fund Investment"
            ]
        ],
        on="cohort_year"
    )
)

cohort_df

,cohort_year,Average SIP Amount,Total Invested,Top Fund,Fund Investment
0,2024,10996.885825,3491125187,102885,100126141
1,2025,13505.209581,30455243,119599,1478507


In [28]:
cohort_df.to_csv(
    "../reports/investor_cohort_analysis.csv",
    index=False
)

In [29]:
import plotly.express as px

fig = px.bar(
    cohort_df,
    x="cohort_year",
    y="Total Invested",
    text="Total Invested",
    color="Total Invested",
    title="Investor Cohort Analysis"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Cohort Year",
    yaxis_title="Total Investment (₹)"
)

fig.show()

In [30]:
sip = transactions[
    transactions["transaction_type"] == "SIP"
].copy()

sip["transaction_date"] = pd.to_datetime(sip["transaction_date"])

sip = sip.sort_values(
    ["investor_id", "transaction_date"]
)

sip.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,cohort_year
19621,INV000001,2024-11-04,120505,SIP,44856,Haryana,Gurugram,T30,36-45,Male,19.9,UPI,Verified,2024
24448,INV000001,2025-01-19,125497,SIP,3090,Haryana,Gurugram,T30,36-45,Male,19.9,Cheque,Pending,2024
5650,INV000002,2024-03-29,149322,SIP,2830,Maharashtra,Pune,T30,46-55,Male,24.0,Mandate,Verified,2024
16803,INV000002,2024-09-21,120841,SIP,2354,Maharashtra,Pune,T30,46-55,Male,24.0,Mandate,Verified,2024
31881,INV000002,2025-05-17,119094,SIP,2690,Maharashtra,Pune,T30,46-55,Male,24.0,Cheque,Verified,2024


In [31]:
sip_count = (
    sip.groupby("investor_id")
       .size()
       .reset_index(name="SIP_Count")
)

eligible = sip_count[
    sip_count["SIP_Count"] >= 6
]

eligible.head()

,investor_id,SIP_Count
3,INV000004,6
7,INV000008,6
9,INV000010,6
10,INV000011,7
11,INV000012,8


In [32]:
sip = sip.merge(
    eligible[["investor_id", "SIP_Count"]],
    on="investor_id"
)

sip.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,cohort_year,SIP_Count
0,INV000004,2024-03-16,101208,SIP,960,Punjab,Chandigarh,T30,26-35,Male,20.0,UPI,Verified,2024,6
1,INV000004,2024-04-11,119095,SIP,20602,Punjab,Chandigarh,T30,26-35,Male,20.0,Net Banking,Verified,2024,6
2,INV000004,2024-05-09,120844,SIP,541,Punjab,Chandigarh,T30,26-35,Male,20.0,Mandate,Verified,2024,6
3,INV000004,2024-07-07,148569,SIP,9761,Punjab,Chandigarh,T30,26-35,Male,20.0,UPI,Verified,2024,6
4,INV000004,2025-03-29,149324,SIP,14282,Punjab,Chandigarh,T30,26-35,Male,20.0,Mandate,Verified,2024,6


In [33]:
sip["Gap_Days"] = (
    sip.groupby("investor_id")["transaction_date"]
       .diff()
       .dt.days
)

sip.head(10)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,cohort_year,SIP_Count,Gap_Days
0,INV000004,2024-03-16,101208,SIP,960,Punjab,Chandigarh,T30,26-35,Male,20.0,UPI,Verified,2024,6,NaN
1,INV000004,2024-04-11,119095,SIP,20602,Punjab,Chandigarh,T30,26-35,Male,20.0,Net Banking,Verified,2024,6,26.0
2,INV000004,2024-05-09,120844,SIP,541,Punjab,Chandigarh,T30,26-35,Male,20.0,Mandate,Verified,2024,6,28.0
3,INV000004,2024-07-07,148569,SIP,9761,Punjab,Chandigarh,T30,26-35,Male,20.0,UPI,Verified,2024,6,59.0
4,INV000004,2025-03-29,149324,SIP,14282,Punjab,Chandigarh,T30,26-35,Male,20.0,Mandate,Verified,2024,6,265.0
5,INV000004,2025-05-17,119599,SIP,2110,Punjab,Chandigarh,T30,26-35,Male,20.0,UPI,Verified,2024,6,49.0
6,INV000008,2024-05-27,101206,SIP,8061,Punjab,Amritsar,B30,26-35,Female,18.6,Cheque,Verified,2024,6,NaN
7,INV000008,2024-06-23,102887,SIP,55850,Punjab,Amritsar,B30,26-35,Female,18.6,Net Banking,Verified,2024,6,27.0
8,INV000008,2024-12-05,100025,SIP,425,Punjab,Amritsar,B30,26-35,Female,18.6,UPI,Verified,2024,6,165.0
9,INV000008,2025-01-18,119552,SIP,458,Punjab,Amritsar,B30,26-35,Female,18.6,Cheque,Verified,2024,6,44.0


In [34]:
gap_df = (
    sip.groupby("investor_id")
       .agg(
           SIP_Count=("SIP_Count", "first"),
           Avg_Gap_Days=("Gap_Days", "mean")
       )
       .reset_index()
)

gap_df.head()

,investor_id,SIP_Count,Avg_Gap_Days
0,INV000004,6,85.400000
1,INV000008,6,70.400000
2,INV000010,6,64.800000
3,INV000011,7,40.166667
4,INV000012,8,57.000000


In [35]:
gap_df["Status"] = np.where(
    gap_df["Avg_Gap_Days"] > 35,
    "At-Risk",
    "Regular"
)

gap_df.head()

,investor_id,SIP_Count,Avg_Gap_Days,Status
0,INV000004,6,85.400000,At-Risk
1,INV000008,6,70.400000,At-Risk
2,INV000010,6,64.800000,At-Risk
3,INV000011,7,40.166667,At-Risk
4,INV000012,8,57.000000,At-Risk


In [36]:
gap_df["Status"].value_counts()

Status
At-Risk    1332
Regular      30
Name: count, dtype: int64

In [37]:
gap_df.to_csv(
    "../reports/sip_continuity_report.csv",
    index=False
)

In [38]:
import plotly.express as px

status_count = (
    gap_df["Status"]
    .value_counts()
    .reset_index()
)

status_count.columns = [
    "Status",
    "Investors"
]

fig = px.pie(
    status_count,
    names="Status",
    values="Investors",
    title="SIP Continuity Status"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

In [39]:
fig.write_image(
    "../reports/sip_continuity_chart.png",
    width=1200,
    height=800
)

In [43]:
sharpe_df.head()

,amfi_code,Sharpe Ratio
0,148567,1.448
1,120843,1.307
2,148569,1.235
3,119551,1.208
4,120505,1.180


In [41]:
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [44]:
recommendation_df = sharpe_df.merge(
    fund_master,
    on="amfi_code"
)

recommendation_df.head()

,amfi_code,Sharpe Ratio,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,148567,1.448,Mirae Asset MF,Mirae Asset Large Cap Fund - Regular - Growth,Equity,Large Cap,Regular,2008-04-04,NIFTY 100 TRI,1.46,1.0,500,1000,Gaurav Misra,Moderate,EC01
1,120843,1.307,Kotak Mahindra MF,Kotak Flexicap Fund - Regular - Growth,Equity,Flexi Cap,Regular,2009-08-11,NIFTY 500 TRI,1.45,1.0,500,1000,Pankaj Tibrewal,Moderately High,EC04
2,148569,1.235,Mirae Asset MF,Mirae Asset Tax Saver Fund - Regular - Growth,Equity,ELSS,Regular,2015-12-28,NIFTY 500 TRI,1.60,1.0,500,1000,Neelesh Surana,High,EC05
3,119551,1.208,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
4,120505,1.180,ICICI Prudential MF,ICICI Pru Midcap Fund - Regular - Growth,Equity,Mid Cap,Regular,2004-10-28,NIFTY Midcap 150 TRI,1.36,1.0,500,1000,Sankaran Naren,High,EC02


In [45]:
def recommend_funds(risk):

    result = (
        recommendation_df[
            recommendation_df["risk_grade"].str.lower() == risk.lower()
        ]
        .sort_values(
            "Sharpe",
            ascending=False
        )
        .head(3)
    )

    return result[
        [
            "amfi_code",
            "scheme_name",
            "risk_grade",
            "Sharpe"
        ]
    ]

In [48]:
recommendation_df = sharpe_df.merge(
    fund_master,
    on="amfi_code"
)

recommendation_df.head()

,amfi_code,Sharpe Ratio,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,148567,1.448,Mirae Asset MF,Mirae Asset Large Cap Fund - Regular - Growth,Equity,Large Cap,Regular,2008-04-04,NIFTY 100 TRI,1.46,1.0,500,1000,Gaurav Misra,Moderate,EC01
1,120843,1.307,Kotak Mahindra MF,Kotak Flexicap Fund - Regular - Growth,Equity,Flexi Cap,Regular,2009-08-11,NIFTY 500 TRI,1.45,1.0,500,1000,Pankaj Tibrewal,Moderately High,EC04
2,148569,1.235,Mirae Asset MF,Mirae Asset Tax Saver Fund - Regular - Growth,Equity,ELSS,Regular,2015-12-28,NIFTY 500 TRI,1.60,1.0,500,1000,Neelesh Surana,High,EC05
3,119551,1.208,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
4,120505,1.180,ICICI Prudential MF,ICICI Pru Midcap Fund - Regular - Growth,Equity,Mid Cap,Regular,2004-10-28,NIFTY Midcap 150 TRI,1.36,1.0,500,1000,Sankaran Naren,High,EC02


In [49]:
recommendation_df["risk_category"].value_counts()

risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

In [50]:
def recommend_funds(risk):

    risk = risk.lower()

    if risk == "low":
        allowed = ["Low", "Moderately Low"]

    elif risk == "moderate":
        allowed = ["Moderate", "Moderately High"]

    elif risk == "high":
        allowed = ["High", "Very High"]

    else:
        print("Invalid Risk Level")
        return

    result = (
        recommendation_df[
            recommendation_df["risk_category"].isin(allowed)
        ]
        .sort_values(
            "Sharpe",
            ascending=False
        )
        .head(3)
    )

    return result[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "risk_category",
            "Sharpe"
        ]
    ]

In [52]:
sharpe_df.columns.tolist()

['amfi_code', 'Sharpe Ratio']

In [53]:
recommendation_df.columns.tolist()


['amfi_code',
 'Sharpe Ratio',
 'fund_house',
 'scheme_name',
 'category',
 'sub_category',
 'plan',
 'launch_date',
 'benchmark',
 'expense_ratio_pct',
 'exit_load_pct',
 'min_sip_amount',
 'min_lumpsum_amount',
 'fund_manager',
 'risk_category',
 'sebi_category_code']

In [54]:
import pandas as pd

# Load datasets
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")
sharpe_df = pd.read_csv("../reports/sharpe_ratio.csv")

# Merge datasets
recommendation_df = pd.merge(
    sharpe_df,
    fund_master,
    on="amfi_code",
    how="inner"
)

# Recommendation function
def recommend_funds(risk):

    risk = risk.lower()

    if risk == "low":
        allowed = ["Low", "Moderately Low"]

    elif risk == "moderate":
        allowed = ["Moderate", "Moderately High"]

    elif risk == "high":
        allowed = ["High", "Very High"]

    else:
        print("Please enter Low, Moderate or High")
        return pd.DataFrame()

    result = (
        recommendation_df[
            recommendation_df["risk_category"].isin(allowed)
        ]
        .sort_values(
            "Sharpe Ratio",
            ascending=False
        )
        .head(3)
    )

    return result[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "risk_category",
            "Sharpe Ratio"
        ]
    ]


# =============================
# Interactive Program
# =============================

risk = input("Enter Risk Appetite (Low / Moderate / High): ")

print("\nTop 3 Recommended Funds\n")

print(recommend_funds(risk))


Top 3 Recommended Funds

    amfi_code                                   scheme_name  \
22     120507      ICICI Pru Liquid Fund - Regular - Growth   
33     120844          Kotak Liquid Fund - Regular - Growth   
36     119120  SBI Magnum Gilt Fund - Regular Plan - Growth   

             fund_house risk_category  Sharpe Ratio  
22  ICICI Prudential MF           Low         0.496  
33    Kotak Mahindra MF           Low        -0.089  
36      SBI Mutual Fund           Low        -0.227  


In [55]:
recommend_funds("Low")

,amfi_code,scheme_name,fund_house,risk_category,Sharpe Ratio
22,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Low,0.496
33,120844,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Low,-0.089
36,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Low,-0.227


In [56]:
recommend_funds("Moderate")

,amfi_code,scheme_name,fund_house,risk_category,Sharpe Ratio
0,148567,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Moderate,1.448
1,120843,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,Moderately High,1.307
3,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Moderate,1.208


In [57]:
recommend_funds("High")

,amfi_code,scheme_name,fund_house,risk_category,Sharpe Ratio
2,148569,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset MF,High,1.235
4,120505,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,High,1.180
5,149323,DSP Midcap Fund - Regular - Growth,DSP Mutual Fund,High,1.132


In [63]:
import os

os.listdir("../data/processed")

['01_fund_master.csv',
 '02_nav_history.csv',
 '03_aum_by_fund_house.csv',
 '04_monthly_sip_inflows.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '07_scheme_performance.csv',
 '08_investor_transactions.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv',
 'nav_with_returns.csv']

In [70]:
print(os.listdir("../data/processed"))

['01_fund_master.csv', '02_nav_history.csv', '03_aum_by_fund_house.csv', '04_monthly_sip_inflows.csv', '05_category_inflows.csv', '06_industry_folio_count.csv', '07_scheme_performance.csv', '08_investor_transactions.csv', '09_portfolio_holdings.csv', '10_benchmark_indices.csv', 'nav_with_returns.csv']


In [71]:
import os

for root, dirs, files in os.walk("../"):
    for file in files:
        if "09_portfolio_holdings" in file:
            print(os.path.join(root, file))

../data\processed\09_portfolio_holdings.csv
../data\raw\09_portfolio_holdings.csv


In [72]:
holdings = pd.read_csv("../data/processed/09_portfolio_holdings.csv")

In [73]:
import pandas as pd
import numpy as np
import plotly.express as px

# Load data
holdings = pd.read_csv("../data/processed/09_portfolio_holdings.csv")
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")

# Convert weight to decimal
holdings["weight_decimal"] = holdings["weight_pct"] / 100

# Calculate HHI
hhi_df = (
    holdings.groupby("amfi_code")["weight_decimal"]
    .apply(lambda x: (x**2).sum())
    .reset_index(name="HHI")
)

# Merge with fund details
hhi_df = hhi_df.merge(
    fund_master[
        ["amfi_code", "scheme_name", "fund_house"]
    ],
    on="amfi_code",
    how="left"
)

# Portfolio classification
def classify_hhi(hhi):
    if hhi < 0.10:
        return "Highly Diversified"
    elif hhi < 0.18:
        return "Moderately Diversified"
    else:
        return "Highly Concentrated"

hhi_df["Portfolio Type"] = hhi_df["HHI"].apply(classify_hhi)

# Sort
hhi_df = hhi_df.sort_values(
    "HHI",
    ascending=False
)

display(hhi_df.head(10))

# Save CSV
hhi_df.to_csv(
    "../reports/sector_hhi.csv",
    index=False
)

# Plot
top10 = hhi_df.head(10)

fig = px.bar(
    top10,
    x="scheme_name",
    y="HHI",
    color="HHI",
    text="HHI",
    title="Top 10 Most Concentrated Mutual Funds"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Scheme Name",
    yaxis_title="HHI"
)

fig.update_xaxes(tickangle=-45)

fig.show()

fig.write_image(
    "../reports/sector_hhi_chart.png",
    width=1600,
    height=900
)

,amfi_code,HHI,scheme_name,fund_house,Portfolio Type
11,119092,0.206448,Axis Bluechip Fund - Regular - Growth,Axis Mutual Fund,Highly Concentrated
3,101207,0.200700,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Highly Concentrated
18,119599,0.174751,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Moderately Diversified
4,102885,0.174709,UTI Nifty 50 Index Fund - Regular - Growth,UTI Mutual Fund,Moderately Diversified
7,118632,0.168298,Nippon India Large Cap Fund - Regular - Growth,Nippon India MF,Moderately Diversified
29,148568,0.167930,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,Moderately Diversified
21,120505,0.157570,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Moderately Diversified
22,120506,0.153794,ICICI Pru Value Discovery Fund - Regular - Growth,ICICI Prudential MF,Moderately Diversified
27,125498,0.152414,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Moderately Diversified
23,120841,0.149680,Kotak Bluechip Fund - Regular - Growth,Kotak Mahindra MF,Moderately Diversified


# Advanced Insights

## 1. Historical Risk Analysis (VaR & CVaR)
The Historical Value at Risk (95%) and Conditional Value at Risk (CVaR) analysis identified the funds with the highest downside risk. Funds having more negative VaR and CVaR values indicate a greater probability of experiencing significant daily losses during adverse market conditions. These funds are more suitable for investors with a higher risk appetite.

---

## 2. Rolling Sharpe Ratio Performance
The 90-day Rolling Sharpe Ratio analysis showed that fund performance varies across different market phases. Funds maintaining consistently high Sharpe Ratios demonstrated superior risk-adjusted returns and better portfolio management compared to funds with highly volatile Sharpe trends.

---

## 3. Investor Behaviour Analysis
Investor cohort analysis revealed differences in investment patterns across cohorts. Recent investor cohorts contributed higher average SIP investments and larger overall investment amounts, indicating increasing participation and confidence in mutual fund investments over time.

---

## 4. SIP Continuity Analysis
The SIP continuity analysis identified investors whose average gap between SIP transactions exceeded 35 days. These investors were classified as "At-Risk", helping fund houses identify customers who may discontinue their SIPs and target them with engagement or retention strategies.

---

## 5. Portfolio Concentration Analysis (HHI)
The Herfindahl-Hirschman Index (HHI) highlighted the degree of portfolio concentration across mutual funds. Funds with lower HHI values were more diversified, reducing concentration risk, whereas higher HHI values indicated portfolios heavily invested in a limited number of holdings or sectors, increasing exposure to sector-specific risks.
